# Chavruta.AI — צינור הטמעה מקצה‑ל‑קצה (Lightning AI · GPU)

מחברת אחת שעושה את כל המסלול, לכל קטגוריה בספריית שעדיין לא הוטמעה:

1. **וידוא (לפני הכול)** — משווה את קטגוריות ה‑shards במאגר המקור מול מאגרי ה‑index הקיימים (`chavruta-index-<slug>`).
   מה שכבר פורסם — מדלגים. מה שחסר — נכנס לתור.
2. **הורדה + התאמה ל‑RAG** — מוריד את ה‑shards, מאחד, ומטמיע ב‑**bge-m3** (dense 1024 + sparse מילוני) — שני ערוצי האחזור ההיברידי (D5).
3. **העלאה במקביל** — בזמן שה‑GPU מטמיע את הקטגוריה הבאה, תרד רקע מעלה את תוצאת הקודמת ל‑HF.

כל קטגוריה → מאגר משלה `Yehuda-Rubin/chavruta-index-<slug>` (כמו `chavruta-index-halacha` הקיים), עם 3 הקבצים בשורש.

⚙️ **דרוש:** מכונת Lightning עם **GPU** (T4/A10/L4 מספיק) · אינטרנט · משתנה סביבה `HF_TOKEN` עם הרשאת כתיבה.

### 1. בדיקת GPU

In [ ]:
!nvidia-smi

### 2. התקנות (גרסאות נעוצות — אחרת bge-m3 נשבר)

In [ ]:
!pip install -q "FlagEmbedding==1.3.4" "transformers==4.44.2" "huggingface_hub>=0.23" numpy

### 3. הגדרות

`HF_TOKEN` נקרא ממשתנה סביבה (או `huggingface-cli login` מקומי). דרוש token עם הרשאת **write**.

In [ ]:
import os, getpass
from pathlib import Path
from huggingface_hub import login, HfApi

NAMESPACE   = "Yehuda-Rubin"
SOURCE_REPO = f"{NAMESPACE}/chavruta-torah-mixed"          # shards: <slug>_partNN.jsonl
INDEX_TMPL  = f"{NAMESPACE}/chavruta-index-{{slug}}"       # מאגר יעד לכל קטגוריה

# ── טוקן כתיבה ל-HF ─────────────────────────────────────────────────────────
# מקור הטוקן: https://huggingface.co/settings/tokens  →  New token  →  הרשאת "Write".
# ⚠️ טוקן Read ייכשל ב-create_repo עם 403 Forbidden! חייב Write.
# אם HF_TOKEN מוגדר כמשתנה סביבה / Secret ב-Lightning — ייקרא לבד; אחרת תתבקש להדביק.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("HF *write* token (hf_...): ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)

# אימות הרשאת כתיבה — נכשל מיד אם הטוקן הוא read (לפני שמבזבזים הטמעה)
_who  = HfApi().whoami(token=HF_TOKEN)
_role = ((_who.get("auth") or {}).get("accessToken") or {}).get("role")
print(f"auth   : {_who.get('name')}  | role: {_role}")
if _role == "read":
    raise SystemExit("❌ הטוקן הוא READ בלבד — create_repo ייכשל ב-403. "
                     "צור טוקן Write: https://huggingface.co/settings/tokens")

# גודל אצווה להטמעה. זהה ל‑Job של Nebius, רק גדול בהרבה כי H100 מחזיק את זה בקלות.
# H100 80GB → 1024 (אפשר לנסות גם יותר). GPU קטן (T4/L4/A10) → הורד ל‑256/128.
BATCH    = 1024
MAX_LEN  = 512
CKPT_EVERY = 20_000     # checkpoint כל N צ'אנקים (התאוששות אם ה‑session נופל)

# מקורות "קובץ יחיד" שאינם _part shards — slug → רשימת קבצים במאגר המקור.
# תנ"ך/גמרא/משנה: הקבצים האלה נשאבו מחדש עם מפרשים מלאים, ונטמיעים מחדש כאן.
EXTRA_SOURCES = {
    "tanakh":  ["all_chunks_full.json"],
    "gemara":  ["gemara_chunks.jsonl"],
    "mishnah": ["mishnah_chunks.json"],
    "yerushalmi": ["yerushalmi_chunks.jsonl"],   # Talmud Yerushalmi — new tier talmud_yerushalmi
}
# קטגוריות שיוטמעו מחדש *גם אם* כבר קיים להן index (כי הרחבנו את המפרשים).
FORCE_REEMBED = {"tanakh", "gemara", "mishnah"}

WORK = Path("work"); WORK.mkdir(exist_ok=True)
INDEX_FILES = ("corpus_vectors.npy", "corpus_sparse.jsonl", "corpus_meta.jsonl")
print("source :", SOURCE_REPO)
print("target :", INDEX_TMPL.format(slug="<slug>"))
print("batch  :", BATCH, "| force:", ", ".join(sorted(FORCE_REEMBED)))

### 4. וידוא — מה עוד צריך להטמיע, ומה כבר קיים

סורק את קידומות ה‑shards במקור (`<slug>_part`), ולכל אחת בודק אם מאגר ה‑index שלה קיים ומכיל את 3 הקבצים. מה שלא — נכנס לתור `pending`.

In [ ]:
import re
from huggingface_hub import HfApi

api = HfApi()

# כל קידומות ה‑shards במקור → רשימת slugs מועמדים + גודל בייטים (למיון)
info = api.repo_info(SOURCE_REPO, repo_type="dataset", files_metadata=True, token=HF_TOKEN)
fsize = {s.rfilename: (s.size or 0) for s in info.siblings}

shards_by_slug = {}
for f in fsize:
    m = re.match(r"(.+)_part\d+\.jsonl$", f)
    if m:
        shards_by_slug.setdefault(m.group(1), []).append(f)

# הוסף מקורות קובץ‑יחיד (תנ"ך/גמרא/משנה מקבצי ה‑chunks המלאים)
for slug, files in EXTRA_SOURCES.items():
    shards_by_slug[slug] = list(files)


def src_bytes(slug):
    return sum(fsize.get(f, 0) for f in shards_by_slug[slug])


def index_done(slug):
    """True אם מאגר ה‑index של הקטגוריה קיים ומכיל את כל 3 הקבצים (כולל sparse)."""
    repo = INDEX_TMPL.format(slug=slug)
    try:
        have = set(api.list_repo_files(repo, repo_type="dataset", token=HF_TOKEN))
    except Exception:
        return False
    return all(f in have for f in INDEX_FILES)


pending, done = [], []
for slug in shards_by_slug:
    if slug in FORCE_REEMBED or not index_done(slug):
        pending.append(slug)
    else:
        done.append(slug)

pending.sort(key=src_bytes)        # מהקטן לגדול — מתחילים בקל ומסיימים בגמרא
done.sort()

print(f"סה\"כ {len(shards_by_slug)} קטגוריות עם מקור\n")
print(f"✅ כבר מוטמע מלא ({len(done)}): {', '.join(done) or '—'}")
print(f"⏳ ממתין להטמעה ({len(pending)}) — לפי סדר ביצוע (קטן→גדול):")
for slug in pending:
    tag = "  (re-embed)" if slug in FORCE_REEMBED else ""
    print(f"     {slug:18s} {src_bytes(slug)/1e6:8.1f} MB מקור{tag}")

### 5. פונקציות עזר — מיזוג, הטמעה, שמירה, העלאה

In [ ]:
import json, time, pickle
from pathlib import Path
import numpy as np
from huggingface_hub import hf_hub_download, create_repo


def merge_shards(slug):
    """מוריד ומאחד את כל מקורות הקטגוריה → רשימת chunks. תומך גם ב‑.jsonl (שורות) וגם ב‑.json (אובייקט עם 'chunks')."""
    files = sorted(shards_by_slug[slug])
    chunks = []
    for fn in files:
        local = hf_hub_download(repo_id=SOURCE_REPO, filename=fn,
                                repo_type="dataset", token=HF_TOKEN)
        if fn.endswith(".jsonl"):
            for line in Path(local).open(encoding="utf-8"):
                if line.strip():
                    chunks.append(json.loads(line))
        else:  # אובייקט JSON שלם: {"chunks": [...]} או רשימה חשופה
            raw = json.loads(Path(local).read_text(encoding="utf-8"))
            chunks.extend(raw["chunks"] if isinstance(raw, dict) else raw)
    print(f"   [{slug}] merged {len(files)} source(s) → {len(chunks):,} chunks")
    return chunks


_model = None
def get_model():
    """טוען את bge-m3 פעם אחת וממחזר בין קטגוריות."""
    global _model
    if _model is None:
        import torch
        from FlagEmbedding import BGEM3FlagModel
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        if dev == "cpu":
            print("⚠️  אין GPU — זה יהיה איטי מאוד")
        _model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=(dev == "cuda"), device=dev)
    return _model


def embed_category(slug, chunks):
    """מטמיע dense+sparse עם checkpoint לקטגוריה, שומר ל‑work/<slug>/, מחזיר את התיקיה."""
    out = WORK / slug; out.mkdir(parents=True, exist_ok=True)
    if all((out / f).exists() for f in INDEX_FILES):
        print(f"   [{slug}] כבר מוטמע מקומית — מדלג הטמעה")
        return out
    docs  = [c["document"] for c in chunks]
    ids   = [c["id"] for c in chunks]
    metas = [c["metadata"] for c in chunks]

    ckpt = out / "embed_ckpt.pkl"
    if ckpt.exists():
        st = pickle.loads(ckpt.read_bytes())
        dense_parts, sparse_rows, start = st["dense"], st["sparse"], st["done"]
        print(f"   [{slug}] ממשיך מ‑{start:,}")
    else:
        dense_parts, sparse_rows, start = [], [], 0

    model = get_model()
    t0 = time.time()
    for s in range(start, len(docs), BATCH):
        enc = model.encode(docs[s:s + BATCH], batch_size=BATCH, max_length=MAX_LEN,
                           return_dense=True, return_sparse=True, return_colbert_vecs=False)
        dense_parts.append(np.asarray(enc["dense_vecs"], dtype="float32"))
        for w in enc["lexical_weights"]:
            sparse_rows.append({int(t): float(v) for t, v in dict(w).items()})
        done = min(s + BATCH, len(docs))
        if done % CKPT_EVERY < BATCH or done == len(docs):
            ckpt.write_bytes(pickle.dumps({"dense": dense_parts, "sparse": sparse_rows, "done": done}))
        rate = done / max(time.time() - t0, 1e-9)
        eta = (len(docs) - done) / max(rate, 1e-9) / 60
        print(f"   [{slug}] {done:,}/{len(docs):,}  ({rate:.0f}/s, ETA {eta:.0f}m)", end="\r")

    vecs = np.vstack(dense_parts)
    norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
    vecs = vecs / norms

    np.save(out / "corpus_vectors.npy", vecs)
    with (out / "corpus_sparse.jsonl").open("w", encoding="utf-8") as f:
        for i, row in enumerate(sparse_rows):
            f.write(json.dumps({"i": i, "sparse": row}) + "\n")
    with (out / "corpus_meta.jsonl").open("w", encoding="utf-8") as f:
        for i, (cid, doc, meta) in enumerate(zip(ids, docs, metas)):
            f.write(json.dumps({"i": i, "id": cid, "document": doc, "metadata": meta},
                               ensure_ascii=False) + "\n")
    ckpt.unlink(missing_ok=True)
    print(f"\n   [{slug}] ✅ הוטמע {vecs.shape[0]:,}×{vecs.shape[1]} ב‑{(time.time()-t0)/60:.1f}m")
    return out


def publish(slug, out_dir):
    """מעלה את 3 קבצי ה‑index ל‑chavruta-index-<slug> (יוצר את המאגר אם צריך)."""
    repo = INDEX_TMPL.format(slug=slug)
    create_repo(repo, repo_type="dataset", exist_ok=True, token=HF_TOKEN)
    for fn in INDEX_FILES:
        p = out_dir / fn
        mb = p.stat().st_size / 1e6
        print(f"   ⬆️  [{slug}] {mb:8.1f} MB  {fn}", flush=True)
        api.upload_file(path_or_fileobj=str(p), path_in_repo=fn,
                        repo_id=repo, repo_type="dataset", token=HF_TOKEN)
    print(f"   ✅ [{slug}] פורסם → https://huggingface.co/datasets/{repo}", flush=True)

### 6. הצינור — סדרתי, אחד בכל פעם (קטן→גדול)

בלי מקבילות. כל קטגוריה מסיימת את **כל** המסלול לפני שמתחילים את הבאה:

**הורד shards → הטמע (bge-m3) → העלה ל‑HF → פנה את הדיסק → הבא בתור.**

הסדר מהקטן לגדול (כפי שמוצג למעלה) — מתחילים בקטגוריות הקלות (reference, מוסר...) ומסיימים בגדולות (גמרא). כל קטגוריה שמסתיימת כבר חיה ב‑HF, וגם אם ה‑session ייפול אחריה — לא מאבדים אותה. כשל בקטגוריה אחת לא עוצר את השאר.

In [ ]:
import shutil, traceback

CLEANUP = True          # מחק work/<slug> אחרי העלאה מוצלחת (חוסך דיסק; הקובץ כבר ב‑HF)
results = {"published": [], "failed": []}

for i, slug in enumerate(pending, 1):
    print(f"\n{'='*55}\n▶️  [{i}/{len(pending)}] {slug}\n{'='*55}", flush=True)
    try:
        chunks  = merge_shards(slug)         # 1) הורד + מזג
        out_dir = embed_category(slug, chunks)  # 2) הטמע (dense+sparse)
        del chunks
        publish(slug, out_dir)               # 3) העלה ל‑HF (מחליף את הקיים)
        results["published"].append(slug)
        if CLEANUP:                          # 4) פנה דיסק לפני הבא
            shutil.rmtree(out_dir, ignore_errors=True)
    except Exception:
        print(f"   ❌ [{slug}] נכשל:\n{traceback.format_exc()}", flush=True)
        results["failed"].append(slug)

print(f"\n{'='*55}")
print(f"✅ פורסמו ({len(results['published'])}): {results['published']}")
if results["failed"]:
    print(f"❌ נכשלו (הרץ שוב את התא — מדלג על מה שכבר הועלה): {results['failed']}")

### 7. בחזרה במחשב שלך — משיכה לתוך Qdrant המקומי

לכל קטגוריה שפורסמה, מושכים את האינדקס הבנוי ומוסיפים (`--append`) לתוך ה‑collection המקומי:

```powershell
# לדוגמה — מדרש + תוספתא:
python scripts/bootstrap_rag.py --repo Yehuda-Rubin/chavruta-index-midrash  --out out_midrash  --append --profile local
python scripts/bootstrap_rag.py --repo Yehuda-Rubin/chavruta-index-tosefta  --out out_tosefta  --append --profile local
```

ההיברידי נדלק אוטומטית — הטוען מזהה את `corpus_sparse.jsonl`.